In [ ]:
# Import Libraries
from pathlib import Path
from io import StringIO
import sys

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)

In [ ]:
# Dataset location and loading

# GitHub raw URL for the processed sample dataset
github_url = (
    "https://raw.githubusercontent.com/"
    "dhanya-harish/ASPICE-Readiness-Framework/"
    "main/data/processed/"
    "aspice_sprint_metrics_processed_sample.csv"
)

# Local path, assuming this notebook is stored inside notebooks/
local_path = Path(
    "../data/processed/aspice_sprint_metrics_processed_sample.csv"
)

try:
    df = pd.read_csv(github_url)
    dataset_source = "GitHub repository"
    print("Dataset successfully loaded from GitHub.")
except Exception as github_error:
    print("Could not load the dataset from GitHub.")
    print("GitHub error:", github_error)

    if local_path.exists():
        df = pd.read_csv(local_path)
        dataset_source = str(local_path)
        print("Dataset successfully loaded from the local repository.")
    else:
        raise FileNotFoundError(
            "The dataset could not be found on GitHub or at the local path. "
            "Please verify that the CSV file exists in data/processed/."
        )

print("\nDataset source:", dataset_source)

In [ ]:
# Dataset preview: df.head()

Dataset preview: df.head()

In [ ]:
# Dataset dimensions: df.shape


rows, columns = df.shape

print("Dataset Dimensions")
print("------------------")
print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")
print(f"Shape: {df.shape}")

In [ ]:
# Column names

print("Dataset Columns")
print("---------------")

for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

In [ ]:
# Dataset information: df.info()

buffer = StringIO()
df.info(buf=buffer)

dataset_information = buffer.getvalue()
print(dataset_information)

In [ ]:
# Descriptive statistics: df.describe()

numeric_summary = df.describe(
    include="number"
).transpose()

display(numeric_summary)

In [ ]:
# Categorical-variable summary

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

if len(categorical_columns) > 0:
    categorical_summary = df[
        categorical_columns
    ].describe().transpose()

    display(categorical_summary)
else:
    print("No categorical columns were found.")

In [ ]:
# Missing-values summary

missing_values = pd.DataFrame(
    {
        "Missing Values": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        ).round(2),
    }
)

missing_values = missing_values.sort_values(
    by="Missing Values",
    ascending=False,
)

display(missing_values)

In [ ]:
# Duplicate observations

duplicate_count = df.duplicated().sum()

print("Duplicate-Record Check")
print("----------------------")
print(f"Number of duplicated rows: {duplicate_count}")

In [ ]:
# Readiness-level distribution table

target_column = "aspice_readiness_level"

if target_column not in df.columns:
    raise KeyError(
        f"The expected target column '{target_column}' "
        "is not available in the dataset."
    )

readiness_distribution = (
    df[target_column]
    .value_counts()
    .sort_index()
    .rename_axis("ASPICE Readiness Level")
    .reset_index(name="Number of Observations")
)

readiness_distribution["Percentage"] = (
    readiness_distribution["Number of Observations"]
    / len(df)
    * 100
).round(2)

display(readiness_distribution)

In [ ]:
# Readiness-level distribution chart

# This chart is automatically saved in your repository's images/ folder.

images_directory = Path("../images")
images_directory.mkdir(
    parents=True,
    exist_ok=True,
)

readiness_counts = (
    df[target_column]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 5))
readiness_counts.plot(kind="bar")

plt.title("Distribution of ASPICE Readiness Levels")
plt.xlabel("ASPICE Readiness Level")
plt.ylabel("Number of Observations")
plt.xticks(rotation=0)
plt.tight_layout()

readiness_chart_path = (
    images_directory
    / "aspice_readiness_level_distribution.png"
)

plt.savefig(
    readiness_chart_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Chart saved to:",
    readiness_chart_path.resolve(),
)

In [ ]:
# Readiness-score distribution

score_column = "aspice_readiness_score"

if score_column in df.columns:
    plt.figure(figsize=(8, 5))

    plt.hist(
        df[score_column].dropna(),
        bins=12,
        edgecolor="black",
    )

    plt.title("Distribution of ASPICE Readiness Scores")
    plt.xlabel("ASPICE Readiness Score")
    plt.ylabel("Frequency")
    plt.tight_layout()

    score_chart_path = (
        images_directory
        / "aspice_readiness_score_distribution.png"
    )

    plt.savefig(
        score_chart_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Chart saved to:",
        score_chart_path.resolve(),
    )
else:
    print(
        f"Column '{score_column}' was not found."
    )

In [ ]:
# Process-area distribution

process_area_column = "process_area"

if process_area_column in df.columns:
    process_area_counts = (
        df[process_area_column]
        .value_counts()
        .sort_index()
    )

    display(
        process_area_counts
        .rename("Number of Observations")
        .to_frame()
    )

    plt.figure(figsize=(8, 5))
    process_area_counts.plot(kind="bar")

    plt.title("Distribution of ASPICE Process Areas")
    plt.xlabel("ASPICE Process Area")
    plt.ylabel("Number of Observations")
    plt.xticks(rotation=0)
    plt.tight_layout()

    process_area_chart_path = (
        images_directory
        / "aspice_process_area_distribution.png"
    )

    plt.savefig(
        process_area_chart_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Chart saved to:",
        process_area_chart_path.resolve(),
    )
else:
    print(
        f"Column '{process_area_column}' was not found."
    )

In [ ]:
# Correlation matrix
numeric_df = df.select_dtypes(include="number")

correlation_matrix = numeric_df.corr()

display(correlation_matrix.round(2))

# Correlation matrix chart
plt.figure(figsize=(14, 11))

correlation_image = plt.imshow(
    correlation_matrix,
    aspect="auto",
    vmin=-1,
    vmax=1,
)

plt.colorbar(
    correlation_image,
    label="Correlation Coefficient",
)

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90,
)

plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
)

plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()

correlation_chart_path = (
    images_directory
    / "correlation_matrix.png"
)

plt.savefig(
    correlation_chart_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    "Chart saved to:",
    correlation_chart_path.resolve(),
)

In [ ]:
# Compliance-indicator summary
compliance_columns = [
    "bp_SWE_1_compliance",
    "bp_SWE_4_compliance",
    "bp_SWE_5_compliance",
    "bp_SUP_1_compliance",
]

available_compliance_columns = [
    column
    for column in compliance_columns
    if column in df.columns
]

if available_compliance_columns:
    compliance_summary = pd.DataFrame(
        {
            "Compliance Count": df[
                available_compliance_columns
            ].sum(),
            "Compliance Rate": (
                df[available_compliance_columns]
                .mean()
                * 100
            ).round(2),
        }
    )

    display(compliance_summary)
else:
    print(
        "No compliance-indicator columns were found."
    )

In [ ]:
# Dataset quality summary
dataset_quality_summary = pd.DataFrame(
    {
        "Measure": [
            "Number of observations",
            "Number of variables",
            "Missing values",
            "Duplicate rows",
            "Numeric variables",
            "Categorical variables",
            "Readiness classes",
        ],
        "Value": [
            len(df),
            len(df.columns),
            int(df.isna().sum().sum()),
            int(df.duplicated().sum()),
            len(
                df.select_dtypes(
                    include="number"
                ).columns
            ),
            len(
                df.select_dtypes(
                    include=["object", "category"]
                ).columns
            ),
            df[target_column].nunique(),
        ],
    }
)

display(dataset_quality_summary)

In [ ]:
# Save summaries as CSV files
reports_directory = Path("../reports")
reports_directory.mkdir(
    parents=True,
    exist_ok=True,
)

numeric_summary.to_csv(
    reports_directory
    / "numeric_descriptive_statistics.csv"
)

missing_values.to_csv(
    reports_directory
    / "missing_values_summary.csv"
)

readiness_distribution.to_csv(
    reports_directory
    / "readiness_level_distribution.csv",
    index=False,
)

dataset_quality_summary.to_csv(
    reports_directory
    / "dataset_quality_summary.csv",
    index=False,
)

print(
    "Summary files saved in:",
    reports_directory.resolve(),
)

In [ ]:
# Final reproducibility statement
print(
    "Dataset exploration completed successfully.\n"
    f"Source: {dataset_source}\n"
    f"Observations: {df.shape[0]}\n"
    f"Variables: {df.shape[1]}\n"
    f"Missing values: {df.isna().sum().sum()}\n"
    f"Duplicate rows: {df.duplicated().sum()}\n"
    f"Target variable: {target_column}"
)